# 📁 PDB File Analysis

---

## Learning Objectives

- Understand PDB file format in detail
- Parse header information (resolution, R-factor, method)
- Extract atomic coordinates
- Analyze chains, ligands, and heteroatoms

---

## PDB File Structure

```
┌─────────────────────────────────────────────────────────────────────────┐
│                         PDB FILE SECTIONS                               │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│   HEADER     ─→ Title, classification, date, PDB ID                     │
│   TITLE      ─→ Descriptive title of the structure                      │
│   COMPND     ─→ Compound (protein) information                          │
│   SOURCE     ─→ Organism source                                         │
│   KEYWDS     ─→ Keywords for searching                                  │
│   EXPDTA     ─→ Experimental method (X-RAY, NMR, CRYO-EM)              │
│   AUTHOR     ─→ Authors                                                 │
│   REMARK     ─→ Comments and experimental details                       │
│   DBREF      ─→ Database cross-references (UniProt, etc.)              │
│   SEQRES     ─→ Primary sequence                                        │
│   HET/HETNAM ─→ Non-standard residues (ligands)                        │
│   HELIX/SHEET─→ Secondary structure assignments                         │
│   CRYST1     ─→ Unit cell parameters                                    │
│   ATOM/HETATM─→ Atomic coordinates ← Main data!                        │
│   CONECT     ─→ Connectivity (bonds) for ligands                        │
│   END        ─→ End of file                                             │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

---

## ATOM Record Format

```
ATOM      1  N   ALA A   1      11.104   6.134  -6.504  1.00 41.88           N
│         │  │   │   │   │      │        │       │      │    │              │
│         │  │   │   │   │      │        │       │      │    │              └─ Element
│         │  │   │   │   │      │        │       │      │    └─ B-factor
│         │  │   │   │   │      │        │       │      └─ Occupancy
│         │  │   │   │   │      │        │       └─ Z coordinate
│         │  │   │   │   │      │        └─ Y coordinate
│         │  │   │   │   │      └─ X coordinate
│         │  │   │   │   └─ Residue number
│         │  │   │   └─ Chain ID
│         │  │   └─ Residue name
│         │  └─ Atom name
│         └─ Atom serial number
└─ Record type

Column positions (1-indexed):
┌─────────┬───────────┬───────────────────────────────┐
│ Columns │ Field     │ Description                   │
├─────────┼───────────┼───────────────────────────────┤
│  1-6    │ Record    │ "ATOM  " or "HETATM"          │
│  7-11   │ Serial    │ Atom serial number            │
│ 13-16   │ Name      │ Atom name (e.g., CA, CB, N)   │
│ 17      │ AltLoc    │ Alternate location indicator  │
│ 18-20   │ ResName   │ Residue name (e.g., ALA, GLY) │
│ 22      │ ChainID   │ Chain identifier              │
│ 23-26   │ ResSeq    │ Residue sequence number       │
│ 31-38   │ X         │ X coordinate (Å)              │
│ 39-46   │ Y         │ Y coordinate (Å)              │
│ 47-54   │ Z         │ Z coordinate (Å)              │
│ 55-60   │ Occupancy │ Occupancy (0.0-1.0)           │
│ 61-66   │ TempFactor│ B-factor (Å²)                 │
│ 77-78   │ Element   │ Element symbol                │
└─────────┴───────────┴───────────────────────────────┘
```

In [ ]:
def parse_atom_line(line):
    """
    Parse a single ATOM/HETATM line from PDB file.
    
    Returns dictionary with all atom properties.
    """
    return {
        'record_type': line[0:6].strip(),
        'serial': int(line[6:11]),
        'atom_name': line[12:16].strip(),
        'alt_loc': line[16].strip(),
        'res_name': line[17:20].strip(),
        'chain_id': line[21],
        'res_seq': int(line[22:26]),
        'x': float(line[30:38]),
        'y': float(line[38:46]),
        'z': float(line[46:54]),
        'occupancy': float(line[54:60]),
        'temp_factor': float(line[60:66]),
        'element': line[76:78].strip() if len(line) > 76 else ''
    }

# Test with example line
example_line = "ATOM      1  N   ALA A   1      11.104   6.134  -6.504  1.00 41.88           N"
atom = parse_atom_line(example_line)

print("Parsed ATOM record:")
for key, value in atom.items():
    print(f"  {key}: {value}")

In [ ]:
class PDBParser:
    """
    Complete PDB file parser.
    """
    
    def __init__(self, filename=None):
        self.header = {}
        self.atoms = []
        self.hetatoms = []
        self.chains = set()
        self.ligands = []
        
        if filename:
            self.parse(filename)
    
    def parse(self, filename):
        """Parse PDB file."""
        with open(filename, 'r') as f:
            for line in f:
                self._parse_line(line)
    
    def _parse_line(self, line):
        """Parse a single line based on record type."""
        record = line[0:6]
        
        if record == 'HEADER':
            self.header['classification'] = line[10:50].strip()
            self.header['date'] = line[50:59].strip()
            self.header['pdb_id'] = line[62:66].strip()
            
        elif record == 'TITLE ':
            title = line[10:80].strip()
            self.header['title'] = self.header.get('title', '') + title
            
        elif record == 'EXPDTA':
            self.header['method'] = line[10:80].strip()
            
        elif 'REMARK   2' in line and 'RESOLUTION' in line:
            try:
                # Extract resolution value
                parts = line.split()
                for i, part in enumerate(parts):
                    if part == 'RESOLUTION.':
                        self.header['resolution'] = float(parts[i+1])
            except:
                pass
                
        elif record == 'CRYST1':
            self.header['cell'] = {
                'a': float(line[6:15]),
                'b': float(line[15:24]),
                'c': float(line[24:33]),
                'alpha': float(line[33:40]),
                'beta': float(line[40:47]),
                'gamma': float(line[47:54]),
                'space_group': line[55:66].strip()
            }
            
        elif record == 'ATOM  ':
            atom = parse_atom_line(line)
            self.atoms.append(atom)
            self.chains.add(atom['chain_id'])
            
        elif record == 'HETATM':
            hetatom = parse_atom_line(line)
            self.hetatoms.append(hetatom)
            # Track unique ligands
            lig_id = (hetatom['res_name'], hetatom['chain_id'], hetatom['res_seq'])
            if lig_id not in self.ligands and hetatom['res_name'] != 'HOH':
                self.ligands.append(lig_id)
    
    def get_ca_atoms(self, chain=None):
        """Get Cα atoms (backbone trace)."""
        ca_atoms = [a for a in self.atoms if a['atom_name'] == 'CA']
        if chain:
            ca_atoms = [a for a in ca_atoms if a['chain_id'] == chain]
        return ca_atoms
    
    def calculate_centroid(self, atoms=None):
        """Calculate center of mass."""
        if atoms is None:
            atoms = self.atoms
        
        n = len(atoms)
        if n == 0:
            return (0, 0, 0)
        
        x = sum(a['x'] for a in atoms) / n
        y = sum(a['y'] for a in atoms) / n
        z = sum(a['z'] for a in atoms) / n
        
        return (x, y, z)
    
    def summary(self):
        """Print structure summary."""
        print("="*60)
        print(f"PDB: {self.header.get('pdb_id', 'Unknown')}")
        print(f"Title: {self.header.get('title', 'N/A')}")
        print(f"Method: {self.header.get('method', 'N/A')}")
        print(f"Resolution: {self.header.get('resolution', 'N/A')} Å")
        print("="*60)
        print(f"Chains: {sorted(self.chains)}")
        print(f"Total atoms: {len(self.atoms)}")
        print(f"Ligands: {[l[0] for l in self.ligands]}")
        print(f"Waters: {len([h for h in self.hetatoms if h['res_name']=='HOH'])}")
        
# Example usage comment (actual file would be needed)
print("PDBParser class defined.")
print("Usage: parser = PDBParser('1crn.pdb')")
print("       parser.summary()")

---

## Extract Backbone Coordinates

The protein backbone consists of repeating N-Cα-C atoms:

```
                 R₁         R₂         R₃
                 │          │          │
    N-terminus   │          │          │   C-terminus
        │        │          │          │        │
        H₂N ─ Cα ─ C ═ O ─ N ─ Cα ─ C ═ O ─ N ─ Cα ─ COO⁻
              │        │       │        │       │
              H        H       H        H       H
              
    Backbone atoms: N, CA, C, O (repeat)
    Cα trace: Just the CA atoms
```

In [ ]:
def extract_backbone(pdb_lines):
    """
    Extract backbone atoms (N, CA, C, O) from PDB lines.
    """
    backbone_atoms = ['N', 'CA', 'C', 'O']
    backbone = []
    
    for line in pdb_lines:
        if line.startswith('ATOM'):
            atom_name = line[12:16].strip()
            if atom_name in backbone_atoms:
                backbone.append(parse_atom_line(line))
    
    return backbone

def extract_ca_trace(pdb_lines):
    """
    Extract only Cα atoms for backbone trace.
    """
    ca_atoms = []
    
    for line in pdb_lines:
        if line.startswith('ATOM'):
            atom_name = line[12:16].strip()
            if atom_name == 'CA':
                ca_atoms.append(parse_atom_line(line))
    
    return ca_atoms

# Example with sample data
sample_pdb = [
    "ATOM      1  N   ALA A   1      11.104   6.134  -6.504  1.00 41.88           N",
    "ATOM      2  CA  ALA A   1      11.639   6.071  -5.147  1.00 38.56           C",
    "ATOM      3  C   ALA A   1      10.488   5.693  -4.217  1.00 33.32           C",
    "ATOM      4  O   ALA A   1       9.478   6.400  -4.182  1.00 32.00           O",
    "ATOM      5  CB  ALA A   1      12.288   7.389  -4.724  1.00 40.34           C",
    "ATOM      6  N   GLY A   2      10.624   4.591  -3.478  1.00 27.54           N",
    "ATOM      7  CA  GLY A   2       9.591   4.114  -2.565  1.00 22.10           C",
]

ca_trace = extract_ca_trace(sample_pdb)
print(f"Found {len(ca_trace)} Cα atoms:")
for ca in ca_trace:
    print(f"  {ca['res_name']} {ca['res_seq']}: ({ca['x']:.2f}, {ca['y']:.2f}, {ca['z']:.2f})")

---

## Important REMARK Records

| REMARK | Content |
|--------|--------|
| REMARK 2 | Resolution |
| REMARK 3 | Refinement statistics (R-factor, R-free) |
| REMARK 200 | Experimental details |
| REMARK 280 | Crystal conditions |
| REMARK 350 | Biological assembly info |
| REMARK 465 | Missing residues |
| REMARK 500 | Geometry deviations |

In [ ]:
def extract_remarks(pdb_file, remark_number):
    """
    Extract specific REMARK records from PDB file.
    
    Parameters:
        pdb_file: Path to PDB file
        remark_number: REMARK number (e.g., 2 for resolution)
    """
    remarks = []
    search_str = f"REMARK {remark_number:3d}"
    
    with open(pdb_file, 'r') as f:
        for line in f:
            if line.startswith(search_str):
                remarks.append(line.rstrip())
    
    return remarks

print("Function to extract REMARK records defined.")
print("Usage: remarks = extract_remarks('1crn.pdb', 3)  # Get refinement stats")

---

## 🏋️ Exercises

### Exercise 1: Count Atoms per Residue
Write a function to count atoms in each residue type.

### Exercise 2: Extract Ligand Information
Parse HETATM records to identify all non-protein molecules.

### Exercise 3: Calculate B-factor Distribution
Analyze the temperature factor distribution across the structure.

---

## 📚 Resources

- [PDB File Format v3.3](https://www.wwpdb.org/documentation/file-format)
- [RCSB PDB](https://www.rcsb.org/)
- [BioPython PDB Module](https://biopython.org/wiki/The_Biopython_Structural_Bioinformatics_FAQ)